# Hope: Transformer Model V2 - Optimized Training (1HZ75V)

Este notebook entrena el modelo Transformer V2 con optimizaciones de rendimiento y entrenamiento (Scheduler, Early Stopping, Class Weights).

## 1. Setup Dependencies

In [ ]:
!pip install torch onnx numpy pandas

## 2. Define Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import pandas as pd
import math
import os
import copy

class SimpleTransformer(nn.Module):
    def __init__(self, input_dim=5, d_model=32, nhead=4, num_layers=3, max_seq_len=32, dropout=0.1):
        super(SimpleTransformer, self).__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.embedding = nn.Linear(input_dim, d_model)
        
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
        self.dropout = nn.Dropout(p=dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.input_norm(x)
        x = self.embedding(x)
        x = x + self.pe[:, :x.size(1), :]
        x = self.dropout(x)
        x = self.transformer_encoder(x)
        x = torch.mean(x, dim=1)
        x = self.fc(x)
        return self.sigmoid(x)

## 3. Data Preparation

Sube `ticks.csv` al almacenamiento de sesión.

In [ ]:
def load_data_from_csv(csv_path, limit=200000):
    if not os.path.exists(csv_path):
        return None
    df = pd.read_csv(csv_path, header=None, names=['epoch', 'quote'], nrows=limit)
    return df['quote'].values.astype(np.float32)

def prepare_features(prices, seq_len=32):
    returns = np.diff(prices)
    directions = np.sign(returns)
    magnitudes = np.abs(returns)
    streaks = []
    reversals = []
    last_trend_direction = 0
    last_direction = 0
    curr_streak = 0
    ticks_since_reversal = 0
    for d in directions:
        if d == 0: curr_streak = 0
        elif d == last_direction: curr_streak += 1
        else: curr_streak = 1
        if (d == 1 and last_trend_direction == -1) or (d == -1 and last_trend_direction == 1):
            ticks_since_reversal = 1
        elif d != 0: ticks_since_reversal += 1
        if d != 0: last_trend_direction = d
        streaks.append(curr_streak)
        reversals.append(ticks_since_reversal)
        last_direction = d
    
    # Task 4: Vectorized Volatility
    vol = pd.Series(returns).rolling(window=20, min_periods=1).std().fillna(0).values
    
    features = np.stack([directions, magnitudes, streaks, reversals, vol], axis=1)
    x, y = [], []
    for i in range(len(features) - seq_len):
        x.append(features[i:i+seq_len])
        y.append(1.0 if returns[i+seq_len] > 0 else 0.0)
    return torch.from_numpy(np.array(x, dtype=np.float32)), torch.from_numpy(np.array(y, dtype=np.float32)).unsqueeze(1)

## 4. Optimized Training

In [ ]:
csv_path = "ticks.csv"
seq_len = 32
input_dim = 5

prices = load_data_from_csv(csv_path)
if prices is not None:
    x_all, y_all = prepare_features(prices, seq_len)
else:
    x_all = torch.randn(2000, seq_len, input_dim)
    y_all = torch.randint(0, 2, (2000, 1)).float()

split_idx = int(len(x_all) * 0.8)
x_train, x_val = x_all[:split_idx], x_all[split_idx:]
y_train, y_val = y_all[:split_idx], y_all[split_idx:]

# Task 9: Class weights
num_pos = torch.sum(y_train).item()
num_neg = len(y_train) - num_pos
pos_weight_val = (num_neg / num_pos) if num_pos > 0 else 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleTransformer(input_dim=input_dim, max_seq_len=seq_len).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

def weighted_bce_loss(output, target, pos_weight):
    loss = - (pos_weight * target * torch.log(output + 1e-8) + (1 - target) * torch.log(1 - output + 1e-8))
    return torch.mean(loss)

# Task 2: Scheduler & Early Stopping
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
best_val_loss = float('inf')
best_model_state = None
early_stop_patience = 5
patience_counter = 0

for epoch in range(100):
    model.train()
    train_loss = 0
    for i in range(0, len(x_train), 64):
        batch_x, batch_y = x_train[i:i+64].to(device), y_train[i:i+64].to(device)
        optimizer.zero_grad()
        output = model(batch_x)
        loss = weighted_bce_loss(output, batch_y, pos_weight_val)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch_x)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for i in range(0, len(x_val), 64):
            batch_x, batch_y = x_val[i:i+64].to(device), y_val[i:i+64].to(device)
            output = model(batch_x)
            loss = weighted_bce_loss(output, batch_y, pos_weight_val)
            val_loss += loss.item() * len(batch_x)
    
    avg_val_loss = val_loss / len(x_val)
    scheduler.step(avg_val_loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
    
    if epoch % 5 == 0: print(f"Epoch {epoch}, Val Loss: {avg_val_loss:.4f}")
    if patience_counter >= early_stop_patience: break

if best_model_state: model.load_state_dict(best_model_state)
print(f"Best Val Loss: {best_val_loss:.4f}")

## 5. Export to ONNX (Static Batch Size 1)

In [ ]:
dummy_input = torch.randn(1, seq_len, input_dim).to(device)
onnx_path = "model.onnx"
model.eval()
# Task 8: Removed dynamic_axes for static Batch Size 1
torch.onnx.export(
    model, dummy_input, onnx_path,
    export_params=True, opset_version=11, do_constant_folding=True,
    input_names=['input'], output_names=['output'],
    training=torch.onnx.TrainingMode.EVAL
)
try:
    from google.colab import files
    files.download(onnx_path)
except: pass